# 07 · Export data for the Tableau dashboard
Writes flat CSVs to `tableau/` so the dashboard needs no Python. Every file comes from the **held-out test set**, so all scores are out-of-sample.

| File | One row per | Used for |
|---|---|---|
| `scored_encounters.csv` | test encounter | call list, risk distribution, who gets called by group |
| `policy_curve.csv` | call-list size (0–100% in 0.5% steps) | net-benefit curve driven by Tableau parameters |
| `gains_by_decile.csv` | risk decile | readmission rate by decile |
| `sensitivity_grid.csv` | program cost × effect | best % to call heatmap |
| `fairness_by_group.csv`, `shap_importance.csv` | group / feature | copied from Step 5 |

The data is the public, de-identified UCI dataset, so these files are safe to commit and publish on Tableau Public.

In [ ]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd

PROC, RES, OUT = Path("data/processed"), Path("results"), Path("tableau")
OUT.mkdir(exist_ok=True)

pred = pd.read_parquet(PROC / "test_predictions.parquet")
df = pd.read_parquet(PROC / "model_table.parquet")
test_patients = set(pd.read_parquet(PROC / "test_patients.parquet")["patient_nbr"])
test = df[df["patient_nbr"].isin(test_patients)].reset_index(drop=True)
assert len(test) == len(pred) and (test["patient_nbr"].values == pred["patient_nbr"].values).all()
test["p_risk"] = pred["p_risk"].values
N = len(test)
print(f"Test encounters: {N:,}")

## 1. Scored encounters (one row per test discharge)

In [ ]:
# Readable labels for discharge disposition (UCI IDs_mapping.csv)
DISPOSITION = {1: "Home", 2: "Short-term hospital", 3: "Skilled nursing facility", 4: "Intermediate care facility",
               5: "Other inpatient institution", 6: "Home with home health", 7: "Left against medical advice",
               8: "Home with IV provider", 9: "Admitted as inpatient", 10: "Neonate transfer",
               12: "Expected outpatient return", 15: "Swing bed", 16: "Other outpatient institution",
               17: "This institution outpatient", 18: "Not recorded", 22: "Rehab facility", 23: "Long-term care hospital",
               24: "Medicaid nursing facility", 25: "Not mapped", 26: "Unknown", 27: "Federal facility",
               28: "Psychiatric hospital", 29: "Critical access hospital", 30: "Other health care institution"}

out = test.sort_values("p_risk", ascending=False, kind="stable").reset_index(drop=True)
out.insert(0, "risk_rank", np.arange(1, N + 1))
out["rank_share"] = out["risk_rank"] / N                      # 0.0001 = riskiest, 1.0 = lowest risk
out["risk_decile"] = (np.arange(N) * 10 // N) + 1             # 1 = highest risk
out["readmitted_30"] = np.where(out["readmit_30"] == 1, "Readmitted", "Not readmitted")

if "age_num" in out:
    out["age_group"] = pd.cut(out["age_num"], [0, 40, 60, 80, 120], right=False,
                              labels=["Under 40", "40-59", "60-79", "80+"]).astype(str)
if "male" in out:
    out["sex"] = np.where(out["male"] == 1, "Male", "Female")
if "discharge_disposition_id" in out:
    ids = pd.to_numeric(out["discharge_disposition_id"].astype(str), errors="coerce")
    out["discharge_disposition"] = [DISPOSITION.get(int(i), f"ID {int(i)}") if pd.notna(i) else "Unknown" for i in ids]

# Drop log-transformed helper columns: Tableau users read the raw counts
out = out.drop(columns=[c for c in out.columns if c.startswith("log_")])
out.to_csv(OUT / "scored_encounters.csv", index=False)
print(f"scored_encounters.csv: {out.shape[0]:,} rows, {out.shape[1]} columns")
out.head(3)

## 2. Policy curve
Cumulative counts only. Tableau computes net benefit from these with parameters for readmission cost, program cost and effect, so the curve updates live.

In [ ]:
y_sorted = out["readmit_30"].to_numpy()
cum = np.concatenate([[0], np.cumsum(y_sorted)])
shares = np.round(np.arange(0, 100.5, 0.5), 1)
called = np.round(shares / 100 * N).astype(int)
curve = pd.DataFrame({"share_called_pct": shares, "called": called, "readmits_reached": cum[called],
                      "total_encounters": N, "total_readmits": int(y_sorted.sum()), "base_rate": y_sorted.mean()})
curve["recall"] = curve["readmits_reached"] / curve["total_readmits"]
curve["precision"] = np.where(curve["called"] > 0, curve["readmits_reached"] / curve["called"].clip(lower=1), np.nan)
curve.to_csv(OUT / "policy_curve.csv", index=False)
curve.iloc[[0, 10, 20, 40, 100, 200]]

## 3. Gains by decile

In [ ]:
gains = out.groupby("risk_decile").agg(encounters=("readmit_30", "size"), readmissions=("readmit_30", "sum"),
                                       readmit_rate=("readmit_30", "mean"), mean_predicted=("p_risk", "mean")).reset_index()
gains["share_of_all_readmits"] = gains["readmissions"] / gains["readmissions"].sum()
gains["cumulative_capture"] = gains["share_of_all_readmits"].cumsum()
gains.to_csv(OUT / "gains_by_decile.csv", index=False)
gains.round(3)

## 4. Sensitivity grid (readmission cost $15,000)

In [ ]:
def best_policy(cost_readmit, cost_pp, effect):
    k = np.arange(1, N + 1)
    net = effect * np.cumsum(y_sorted) * cost_readmit - k * cost_pp
    i = int(net.argmax())
    return (k[i] / N, net[i] / N * 1000) if net[i] > 0 else (0.0, 0.0)

rows = []
for c in [50, 100, 200, 300, 500]:
    for e in [0.05, 0.10, 0.15, 0.20, 0.30, 0.40]:
        s, nb_ = best_policy(15_000, c, e)
        rows.append({"cost_per_patient": c, "effect_pct": int(e * 100), "best_share_called": s, "net_per_1000": round(nb_)})
grid = pd.DataFrame(rows)
grid.to_csv(OUT / "sensitivity_grid.csv", index=False)
grid.pivot(index="cost_per_patient", columns="effect_pct", values="best_share_called").round(2)

## 5. Copy Step 5 results

In [ ]:
for src, dst in [("05_fairness_by_group.csv", "fairness_by_group.csv"), ("05_shap_importance.csv", "shap_importance.csv")]:
    if (RES / src).exists():
        shutil.copy(RES / src, OUT / dst)
        print(f"Copied {dst}")
    else:
        print(f"Missing results/{src}: rerun notebook 05")

for f in sorted(OUT.glob("*.csv")):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:5.2f} MB")